# Minimal atomic-povray workflow

This notebook is the shortest complete path from an ASE-readable structure to an inline POV-Ray render. Geometry is built once and can be reused while changing styles, camera, lighting, or render settings.

In [ ]:
from os import environ
from pathlib import Path

from IPython.display import Image
from atomic_povray import *

project = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
poscar = project / "tests" / "data" / "fe2o3-012-1x1-relaxed.vasp"
output = project / "hematite_minimal.png"
povray = environ.get("POVRAY", "povray")

In [ ]:
structure = load_structure(poscar)
bond_rules = get_default_bonds(structure)

## Optional bond-rule changes

`get_default_bonds` prints the rule names, half-open distance limits, and boundary-extension directions. Make any structure-specific changes in the next cell, before the expensive geometry build. Metal–metal pairs are not generated by default; add them explicitly when needed.

In [ ]:
# Example for a Pt chain with nearest-neighbor distances below 3.1 Å:
# bond_rules.add(
#     BondRule(
#         "Pt", "Pt", 0.0, 3.1,
#         name="custom:Pt-Pt",
#         extension_mode="symmetric",
#     )
# )
# bond_rules.print_table()

In [ ]:
geometry = build_geometry(
    structure,
    bond_rules=bond_rules,
    bounds=DisplayBounds(
        fractional_ranges=((-2.0, 2.0), (-1.5, 1.5), (0.45, 0.75)),
    ),
)
styled = apply_styles(geometry, StyleConfig())
len(geometry.primary_atoms), len(geometry.extension_atoms), len(geometry.bonds)

In [ ]:
camera = Camera.orthographic(
    direction=(0.0, 100.0, 0.0),
    target=(0.0, 0.0, 21.0),
    up=(0.0, 0.0, 1.0),
    width=21.0,
)
scene = make_scene(
    styled.primitives,
    camera=camera,
    lights=(get_default_light(camera),),
    background=Background(Color(1.0, 1.0, 1.0)),
)

In [ ]:
result = render_scene(
    scene,
    output,
    RenderConfig(
        width=1024,
        height=768,
        quality=5,
        executable=povray,
    ),
)
Image(filename=result.image_path)

Set the `POVRAY` environment variable to the full executable path when POV-Ray is not on `PATH`. On Linux or in the Conda environment this is normally simply `povray`; a common Windows value is `C:\\Program Files\\POV-Ray\\v3.7\\bin\\pvengine64.exe`.